# zvec GPU Benchmark Suite

In [ ]:
# Setup
!rm -rf zvec
!git clone -b sprint-gpu-optimization https://github.com/cluster2600/zvec.git
%cd zvec
!pip install faiss-gpu-cu12 -q

In [ ]:
# Imports
import faiss
import numpy as np
import time
print(f"FAISS GPUs: {faiss.get_num_gpus()}")

In [ ]:
# Test 1: FAISS GPU Flat Index
dim = 128
n_vectors = 100000
vectors = np.random.random((n_vectors, dim)).astype(np.float32)
queries = np.random.random((100, dim)).astype(np.float32)

# CPU
index_cpu = faiss.IndexFlatL2(dim)
index_cpu.add(vectors)
start = time.time()
D_cpu, I_cpu = index_cpu.search(queries, k=10)
cpu_time = time.time() - start

# GPU
gpu_resources = faiss.StandardGpuResources()
index_gpu = faiss.index_cpu_to_gpu(gpu_resources, 0, index_cpu)
start = time.time()
D_gpu, I_gpu = index_gpu.search(queries, k=10)
gpu_time = time.time() - start

print(f"Flat Index:")
print(f"  CPU: {cpu_time:.4f}s")
print(f"  GPU: {gpu_time:.4f}s")
print(f"  Speedup: {cpu_time/gpu_time:.1f}x")

In [ ]:
# Test 2: FAISS IVF-PQ Index
nlist = 100
nprobe = 10

# CPU
index_cpu = faiss.IndexIVFFlat(faiss.IndexFlatL2(dim), dim, nlist)
index_cpu.train(vectors[:10000])
index_cpu.add(vectors)
start = time.time()
D_cpu, I_cpu = index_cpu.search(queries, k=10)
cpu_time = time.time() - start

# GPU
gpu_resources = faiss.StandardGpuResources()
index_gpu = faiss.index_cpu_to_gpu(gpu_resources, 0, index_cpu)
start = time.time()
D_gpu, I_gpu = index_gpu.search(queries, k=10)
gpu_time = time.time() - start

print(f"IVF-PQ Index:")
print(f"  CPU: {cpu_time:.4f}s")
print(f"  GPU: {gpu_time:.4f}s")
print(f"  Speedup: {cpu_time/gpu_time:.1f}x")

In [ ]:
# Test 3: Scale test - 1M vectors
n_vectors = 1000000
vectors = np.random.random((n_vectors, dim)).astype(np.float32)
queries = np.random.random((50, dim)).astype(np.float32)

index_cpu = faiss.IndexFlatL2(dim)
index_cpu.add(vectors)

start = time.time()
D_cpu, I_cpu = index_cpu.search(queries, k=10)
cpu_time = time.time() - start

gpu_resources = faiss.StandardGpuResources()
index_gpu = faiss.index_cpu_to_gpu(gpu_resources, 0, index_cpu)

start = time.time()
D_gpu, I_gpu = index_gpu.search(queries, k=10)
gpu_time = time.time() - start

print(f"1M Vectors:")
print(f"  CPU: {cpu_time:.4f}s")
print(f"  GPU: {gpu_time:.4f}s")
print(f"  Speedup: {cpu_time/gpu_time:.1f}x")

In [ ]:
# Test 4: Batch size comparison
batch_sizes = [1, 10, 50, 100, 500]
print("Batch size benchmark:")
for bs in batch_sizes:
    queries = np.random.random((bs, dim)).astype(np.float32)
    
    start = time.time()
    D, I = index_gpu.search(queries, k=10)
    t = time.time() - start
    
    print(f"  Batch {bs}: {t*1000:.2f}ms ({bs/t:.0f} queries/sec)")

In [ ]:
# Summary
print("\n=== SUMMARY ===")
print(f"GPU: {faiss.get_num_gpus()}x NVIDIA")
print(f"All tests passed!")